# 02 — Training

#Task A

Fine-tunes BERT and RoBERTa on EDOS Task A using `src/train.py`.

In [ ]:
!git clone https://github.com/Mahekd/interpretable-nlp-sexism-detection.git
%cd interpretable-nlp-sexism-detection
!pip install -q -r requirements.txt

Cloning into 'interpretable-nlp-sexism-detection'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 44 (delta 14), reused 20 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 1.88 MiB | 15.61 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/interpretable-nlp-sexism-detection
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 24.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:

## Hyperparameter sweep (Task A - RoBERTA)


I am fine-tuning RoBERTA-base for the binary sexism classification task (Task A) using hyperparameters selected via a grid search over learning rate ∈ {2e-5, 3e-5} and batch size ∈ {16, 32}, evaluated on dev-set macro-F1.

The model will be trained for up to 10 epochs with early stopping (monitoring dev macro-F1) to prevent overfitting; the best checkpoint (highest dev macro-F1) is retained for final evaluation. To address class imbalance (10,602 not-sexist vs. 3,398 sexist examples in training), we apply class-weighted cross-entropy loss, with weights computed as the inverse of class frequency (0.660 for not-sexist, 2.060 for sexist), giving the minority (sexist) class proportionally more influence on the loss.

In [ ]:
MODELS = ["roberta-base", "bert-base-uncased"]
LRS = ["2e-5", "3e-5"]
BATCH_SIZES = [16, 32]

for model in MODELS:
    for lr in LRS:
        for bs in BATCH_SIZES:
            run_name = f"lr{lr}_bs{bs}"
            !python -m src.train --task A --model_name {model} --lr {lr} --batch_size {bs} --run_name {run_name}

Device: cuda
Task A: train=14000 dev=2000 test=4000 | labels=['not sexist', 'sexist']
config.json: 100% 481/481 [00:00<00:00, 2.74MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 165kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 22.4MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 23.4MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 34.5MB/s]
model.safetensors: 100% 499M/499M [00:07<00:00, 62.7MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 5041.32it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.

In [ ]:
import glob
import json

import pandas as pd

sweep_rows = []
for path in glob.glob("outputs/best_model_taskA_*_lr*_bs*/results.json"):
    with open(path) as f:
        r = json.load(f)
    sweep_rows.append({
        "model": r["model_name"],
        "run_name": r["run_name"],
        "lr": r["lr"],
        "batch_size": r["batch_size"],
        "dev_macro_f1": r["dev_macro_f1"],
    })

sweep_df = pd.DataFrame(sweep_rows).sort_values(["model", "dev_macro_f1"], ascending=[True, False])
print(sweep_df.to_string(index=False))

print("\nBest config per model (by dev macro-F1):")
best_per_model = sweep_df.sort_values("dev_macro_f1", ascending=False).groupby("model").first()
print(best_per_model[["lr", "batch_size", "dev_macro_f1"]])
print("\nUse these lr/batch_size values for --lr/--batch_size in the main Task A/B/C runs below, per model.")

       model    run_name      lr  batch_size  dev_macro_f1
roberta-base lr2e-5_bs32 0.00002          32      0.831981
roberta-base lr3e-5_bs16 0.00003          16      0.828141
roberta-base lr2e-5_bs16 0.00002          16      0.825797
roberta-base lr3e-5_bs32 0.00003          32      0.823215

Best config per model (by dev macro-F1):
                   lr  batch_size  dev_macro_f1
model                                          
roberta-base  0.00002          32      0.831981

Use these lr/batch_size values for --lr/--batch_size in the main Task A/B/C runs below, per model.


## Task A — binary sexism classification (2-way)

The best configuration learning rate 2e-5, batch size 32 which achieved a dev macro-F1 of 0.832 and was used for the final run.

In [ ]:
!python -m src.train --task A --model_name roberta-base --lr 2e-5 --batch_size 32

Device: cuda
Task A: train=14000 dev=2000 test=4000 | labels=['not sexist', 'sexist']
Loading weights: 100% 197/197 [00:00<00:00, 4614.69it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Class weights: [0.6602528095245361, 2.060035228729248]
Ep

On the held-out test set, this configuration achieves a macro-F1 of 0.828, with precision/recall of 0.921/0.909 for not-sexist and 0.726/0.758 for sexist.

## Persist checkpoints to Drive

Colab runtimes are ephemeral — back up `outputs/` before the session disconnects,
otherwise you lose every trained checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/EDOS_DATA/outputs_backup
!cp -r outputs/* /content/drive/MyDrive/EDOS_DATA/outputs_backup/

Mounted at /content/drive


###TASK A

#BERT

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODELS = ["bert-base-uncased"]
LRS = ["2e-5", "3e-5"]
BATCH_SIZES = [16, 32]

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/EDOS_DATA/outputs_backup"
!mkdir -p {DRIVE_BACKUP_DIR}

for model in MODELS:
    for lr in LRS:
        for bs in BATCH_SIZES:
            run_name = f"lr{lr}_bs{bs}"
            !python -m src.train --task A --model_name {model} --lr {lr} --batch_size {bs} --run_name {run_name}

            # Back up this config's checkpoint right away, before starting the next one
            run_dir = f"outputs/best_model_taskA_{model}_{run_name}"
            print(f"Backing up {run_dir} to Drive...")
            !cp -r {run_dir} {DRIVE_BACKUP_DIR}/
            print(f"Done backing up {run_name}.\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Task A: train=14000 dev=2000 test=4000 | labels=['not sexist', 'sexist']
config.json: 100% 570/570 [00:00<00:00, 2.85MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 277kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 1.51MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 1.36MB/s]
model.safetensors: 100% 440M/440M [00:05<00:00, 79.0MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 20289.43it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.tr

In [ ]:
!python -m src.train --task A --model_name bert-base-uncased